# AB Testing

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

In [2]:
df = pd.read_csv("data/ab_data.csv")
df.head(10)

,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1
5,936923,2017-01-10 15:20:49.083499,control,old_page,0
6,679687,2017-01-19 03:26:46.940749,treatment,new_page,1
7,719014,2017-01-17 01:48:29.539573,control,old_page,0
8,817355,2017-01-04 17:58:08.979471,treatment,new_page,1
9,839785,2017-01-15 18:11:06.610965,treatment,new_page,1


In [3]:
df.shape

(294478, 5)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 294478 entries, 0 to 294477
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   user_id       294478 non-null  int64
 1   timestamp     294478 non-null  str  
 2   group         294478 non-null  str  
 3   landing_page  294478 non-null  str  
 4   converted     294478 non-null  int64
dtypes: int64(2), str(3)
memory usage: 11.2 MB


In [5]:
df["group"].value_counts()

group
treatment    147276
control      147202
Name: count, dtype: int64

In [6]:
df["landing_page"].value_counts()

landing_page
old_page    147239
new_page    147239
Name: count, dtype: int64

In [7]:
df["converted"].value_counts()

converted
0    259241
1     35237
Name: count, dtype: int64

In [8]:
df.groupby("group")["converted"].mean()

group
control      0.120399
treatment    0.118920
Name: converted, dtype: float64

Null Hypothesis (H₀):
Conversion rate of the new page is equal to the old page.

Alternative Hypothesis (H₁):
Conversion rate of the new page is different from the old page.

In [9]:
df_clean = df[((df["group"]=="control") & (df["landing_page"]=="old_page"))
    | ((df["group"]=="treatment") & (df["landing_page"]=="new_page"))] 

removed rows where the experimental assignment did not match the landing page to avoid bias and invalid comparisons

In [10]:
df_clean.shape

(290585, 5)

In [11]:
df_clean.groupby("group")["converted"].mean()

group
control      0.120386
treatment    0.118807
Name: converted, dtype: float64

In [12]:
control = df_clean[df_clean["group"] == "control"]["converted"]
treatment = df_clean[df_clean["group"] == "treatment"]["converted"]

In [17]:
from statsmodels.stats.proportion import proportions_ztest

In [16]:
!pip install statsmodels

   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/9.6 MB 31.5 MB/s eta 0:00:01
   --------------- ------------------------ 3.7/9.6 MB 15.9 MB/s eta 0:00:01
   -------------------- ------------------- 5.0/9.6 MB 11.1 MB/s eta 0:00:01
   ---------------------- ----------------- 5.5/9.6 MB 9.9 MB/s eta 0:00:01
   ----------------------------- ---------- 7.1/9.6 MB 8.2 MB/s eta 0:00:01
   -------------------------------- ------- 7.9/9.6 MB 7.5 MB/s eta 0:00:01
   --------------------------------- ------ 8.1/9.6 MB 7.0 MB/s eta 0:00:01
   ------------------------------------ --- 8.7/9.6 MB 5.9 MB/s eta 0:00:01
   ---------------------------------------  9.4/9.6 MB 5.4 MB/s eta 0:00:01
   ---------------------------------------- 9.6/9.6 MB 5.2 MB/s  0:00:01

   ---------------------------------------- 0/2 [patsy]
   ---------------------------------------- 0/2 [patsy]
   ---------------------------------------- 0/2 [patsy]
  


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
control = df_clean[df_clean["group"] == "control"]
treatment = df_clean[df_clean["group"] == "treatment"]

conversions = np.array([
    control["converted"].sum(),
    treatment["converted"].sum()
])

nobs = np.array([
    control["converted"].count(),
    treatment["converted"].count()
])

In [23]:
z_stat, p_value = proportions_ztest(conversions, nobs)
p_value

np.float64(0.18965258971881804)